# LeWorldModelRec — Training on Colab T4

Everything runs **locally on the Colab instance** (no Drive mount).

**Pipeline :**
1. Check GPU
2. Clone repo + patch __init__
3. Generate simple pendulum dataset
4. Train LeWorldModelRec (reconstruction MSE + FrequencyLoss FFT + SIGReg)
5. Plot loss curves
6. Visualise reconstructions
7. Download best checkpoint

**Différence vs LeWorldModel (JEPA) :**  
Pas de target encoder EMA — supervision directe en pixel space via MSE reconstruction + frequency loss (FFT).  
Pas de loss perceptuelle VGG16 : la FrequencyLoss FFT suffit à stabiliser le decoder (prévient le collapse) et son coût est négligeable (pure FFT, pas de réseau).

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. GPU check

In [ ]:
import subprocess, torch

print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch to T4 runtime first.'

## 2. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/double-pendule-wolrdmodel.git'
REPO_DIR = '/content/WorldModel'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
from pathlib import Path

Path('models/__init__.py').write_text(
"""from .encoder  import ContextEncoder
from .decoder  import Decoder
from .sigreg   import sigreg_loss
from .losses   import PerceptualLoss, FrequencyLoss
from .jepa.model import LeWorldModel, TransitionPredictor
from .rec.model  import LeWorldModelRec
"""
)
print('models/__init__.py patched.')

## 3. Install dependencies

In [ ]:
!pip install -q Pillow matplotlib numpy
print('Dependencies ready.')

## 4. Generate dataset

Generates **2 000 trajectories × 500 frames** (64×64 px) of a simple pendulum.  
Each trajectory has random initial angle **and** random initial velocity.  
~5-10 min on Colab CPU.

During training, a random **window of 32 frames** is sampled from each 500-frame
trajectory at every epoch.

In [ ]:
from pathlib import Path

DATASET_DIR = 'dataset/pendulum'
N_TRAJ      = 2000
N_FRAMES    = 500
IMG_SIZE    = 64

existing = len(list(Path(DATASET_DIR).glob('traj_*.npz'))) if Path(DATASET_DIR).exists() else 0

if existing >= N_TRAJ:
    print(f'Dataset already present ({existing} trajectories) — skipping generation.')
else:
    print(f'Generating {N_TRAJ} trajectories × {N_FRAMES} frames…')
    from data.generate import generate_dataset
    generate_dataset(
        n_trajectories=N_TRAJ,
        n_frames=N_FRAMES,
        img_size=IMG_SIZE,
        dt=0.05,
        output_dir=DATASET_DIR,
        seed=42,
    )

### Quick dataset preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample = np.load(f'{DATASET_DIR}/traj_0000.npz')
frames = sample['frames']   # (T, H, W, 3)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
fig.patch.set_facecolor('#111')
for ax, idx in zip(axes, np.linspace(0, len(frames)-1, 8, dtype=int)):
    ax.imshow(frames[idx])
    ax.axis('off')
    ax.set_title(f't={idx}', color='white', fontsize=8)
plt.suptitle('Trajectory 0 — 8 frames', color='white')
plt.tight_layout()
plt.show()

## 5. Training

| Hyperparameter | Value | Note |
|---|---|---|
| `embed_dim` | 128 | latent dimension |
| `hidden_dim` | 512 | MLP transition FFN |
| `lam` | 0.5 | SIGReg weight — **aligné avec JEPA** |
| `rec_coef` | 1.0 | poids MSE reconstruction |
| `pred_coef` | 1.0 | poids MSE prédiction rollout |
| `freq_coef` | 0.05 | poids FrequencyLoss FFT — prévient le collapse decoder |
| `rollout_k` | 10 | horizon de prédiction — **aligné avec JEPA** (0.5 s = 10 × dt) |
| `seq_len` | 32 | réduit vs JEPA — chaque step rollout passe par le décodeur (OOM sinon) |
| `batch_size` | 16 | réduit — B×T frames stockées en graph pour backprop |
| `epochs` | 50 | |
| `lr` | 1e-4 | AdamW + warmup 5ep + cosine |

**Asymétrie résiduelle documentée :** `seq_len=32` (AE) vs `seq_len=100` (JEPA) — contrainte mémoire due au décodeur pixel-space.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
EMBED_DIM        = 128
HIDDEN_DIM       = 512
LAM              = 0.5    # SIGReg weight — aligné avec JEPA
REC_COEF         = 1.0    # poids reconstruction MSE
PRED_COEF        = 1.0    # poids prédiction rollout MSE
PERCEPTUAL_COEF  = 0.0    # désactivé — pas de VGG
FREQ_COEF        = 0.05   # FrequencyLoss FFT — prévient le collapse decoder
ROLLOUT_K        = 10     # horizon de prédiction — aligné avec JEPA (0.5 s = 10 × dt)
SEQ_LEN          = 32     # fenêtre tirée aléatoirement — réduit pour éviter OOM décodeur
N_PROJ           = 512
EPOCHS           = 50
BATCH_SIZE       = 16     # réduit — B*T frames stockées en graph
LR               = 1e-4
WEIGHT_DECAY     = 0.05
NUM_WORKERS      = 2
CKPT_DIR         = 'checkpoints'
VIS_DIR          = 'visuals'
RESUME           = None   # set to 'checkpoints/rec/lewm_rec_best.pt' to resume

In [ ]:
import time, math
from pathlib import Path

import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from models.rec.model import LeWorldModelRec
from data.dataset import make_seq_dataloaders

device = torch.device('cuda')
print(f'Training on: {torch.cuda.get_device_name(0)}')

# ── Dataloaders ────────────────────────────────────────────────────────────────
train_loader, val_loader = make_seq_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seq_len=SEQ_LEN,
)
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')

# ── Model ──────────────────────────────────────────────────────────────────────
model = LeWorldModelRec(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    lam=LAM,
    rec_coef=REC_COEF,
    pred_coef=PRED_COEF,
    perceptual_coef=0.0,
    freq_coef=FREQ_COEF,
    n_proj=N_PROJ,
    rollout_k=ROLLOUT_K,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# ── Optimizer & scheduler ──────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, EPOCHS - warmup)
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Evaluate ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    sums = {'loss': 0.0, 'rec_loss': 0.0, 'pred_loss': 0.0, 'freq_loss': 0.0, 'sigreg': 0.0}
    for frames, _ in loader:
        m = model(frames.to(device, non_blocking=True))
        for k in sums:
            sums[k] += m[k].item()
    n = len(loader)
    return {k: v / n for k, v in sums.items()}

# ── Resume ─────────────────────────────────────────────────────────────────────
start_epoch = 1
if RESUME and Path(RESUME).exists():
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch=ckpt['epoch'])
    print(f'Resumed from epoch {ckpt["epoch"]}')

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(VIS_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:

# ── Monitoring : puissance GPU · mémoire · temps ───────────────────────────────
import threading, subprocess

_power_readings = []
_stop_monitor   = threading.Event()

def _monitor_power():
    while not _stop_monitor.is_set():
        try:
            out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
                text=True
            ).strip()
            _power_readings.append(float(out))
        except Exception:
            pass
        time.sleep(2)

torch.cuda.reset_peak_memory_stats()
_monitor_thread    = threading.Thread(target=_monitor_power, daemon=True)
_monitor_thread.start()
_train_start_wall  = time.time()
print("Monitoring démarré : puissance GPU (toutes les 2s), mémoire peak, temps total.")


In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

history = {k: [] for k in [
    'train', 'val',
    'rec', 'pred', 'freq', 'sigreg',
    'val_rec', 'val_pred', 'val_freq', 'val_sigreg',
    'lr',
]}
best_val            = float('inf')
best_epoch          = 1
time_per_epoch      = []
memory_per_epoch_MB = []

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0   = time.time()
    sums = {'loss': 0.0, 'rec_loss': 0.0, 'pred_loss': 0.0, 'freq_loss': 0.0, 'sigreg': 0.0}

    for frames, _ in train_loader:
        frames = frames.to(device, non_blocking=True)
        optimizer.zero_grad()
        m = model(frames)
        m['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        for k in sums:
            sums[k] += m[k].item()

    scheduler.step()
    n          = len(train_loader)
    train_loss = sums['loss'] / n
    val_m      = evaluate(model, val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_m['loss'])
    history['rec'].append(sums['rec_loss'] / n)
    history['pred'].append(sums['pred_loss'] / n)
    history['freq'].append(sums['freq_loss'] / n)
    history['sigreg'].append(sums['sigreg'] / n)
    history['val_rec'].append(val_m['rec_loss'])
    history['val_pred'].append(val_m['pred_loss'])
    history['val_freq'].append(val_m['freq_loss'])
    history['val_sigreg'].append(val_m['sigreg'])
    history['lr'].append(optimizer.param_groups[0]['lr'])

    elapsed = time.time() - t0
    time_per_epoch.append(round(elapsed, 2))
    memory_per_epoch_MB.append(round(torch.cuda.max_memory_allocated() / 1e6, 1))

    improved = val_m['loss'] < best_val
    if improved:
        best_val   = val_m['loss']
        best_epoch = epoch
        torch.save({
            'epoch':     epoch,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss':  best_val,
            'args': {
                'embed_dim':       EMBED_DIM,
                'hidden_dim':      HIDDEN_DIM,
                'lam':             LAM,
                'rec_coef':        REC_COEF,
                'pred_coef':       PRED_COEF,
                'perceptual_coef': 0.0,
                'freq_coef':       FREQ_COEF,
                'n_proj':          N_PROJ,
                'rollout_k':       ROLLOUT_K,
            },
        }, f'{CKPT_DIR}/lewm_rec_best.pt')

    lr_now = optimizer.param_groups[0]['lr']
    print(
        f'Epoch {epoch:3d}/{EPOCHS}'
        f'  loss={train_loss:.4f}'
        f'  rec={sums["rec_loss"]/n:.4f}'
        f'  pred={sums["pred_loss"]/n:.4f}'
        f'  freq={sums["freq_loss"]/n:.4f}'
        f'  sig={sums["sigreg"]/n:.4f}'
        f'  val={val_m["loss"]:.4f}'
        f'  lr={lr_now:.2e}'
        f'  mem={memory_per_epoch_MB[-1]:.0f}MB'
        f'  {elapsed:.1f}s'
        + ('  <-- best' if improved else '')
    )

print(f'\nBest val loss: {best_val:.4f}  @epoch {best_epoch}  ->  {CKPT_DIR}/lewm_rec_best.pt')

## 6. Loss curves

In [ ]:
epochs_x = range(1, len(history['train']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.patch.set_facecolor('#111')

ax = axes[0]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['train'], color='#4fc3f7', label='train')
ax.plot(epochs_x, history['val'],   color='#ff8a65', label='val')
ax.set_title('Total loss', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

ax = axes[1]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['rec'],  color='#a5d6a7', label='rec MSE')
ax.plot(epochs_x, history['pred'], color='#4fc3f7', label='pred MSE')
ax.set_title('Pixel losses (MSE)', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

ax = axes[2]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['freq'],   color='#ef9a9a', label='frequency (FFT)')
ax.plot(epochs_x, history['sigreg'], color='#ce93d8', label='SIGReg')
ax.set_title('Regularization', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_rec_loss_colab.png', dpi=100, bbox_inches='tight', facecolor='#111')
plt.show()

## 7. Reconstruction visualisation

Avantage clé de l'architecture AE : on peut décoder les latents et voir ce que le modèle a appris.  
- **Ligne 1** : frames originales  
- **Ligne 2** : reconstructions `decode(encode(frame))`  
- **Ligne 3** : prédictions `decode(predictor(encode(frame_t))) ≈ frame_{t+1}`

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

ckpt = torch.load(f'{CKPT_DIR}/lewm_rec_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  (val_loss={ckpt["val_loss"]:.4f})')

frames_batch, _ = next(iter(val_loader))
sample  = frames_batch[:1].to(device)            # (1, T, 3, H, W)
T_SHOW  = 8
indices = np.linspace(0, sample.shape[1] - 2, T_SHOW, dtype=int)

with torch.no_grad():
    z        = model.encode(sample)              # (1, T, D)
    rec      = model.decode(z)                   # (1, T, 3, H, W)
    z_pred   = model.predictor(z[:, :-1])        # (1, T-1, D)
    pred_dec = model.decode(z_pred)              # (1, T-1, 3, H, W)

orig_np  = sample[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()
rec_np   = rec[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()
pred_np  = pred_dec[0, indices].permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()

fig, axes = plt.subplots(3, T_SHOW, figsize=(T_SHOW * 2, 7))
fig.patch.set_facecolor('#111')
row_labels = ['Original', 'Reconstruction', 'Prediction (t+1)']
row_data   = [orig_np, rec_np, pred_np]

for row, (label, data) in enumerate(zip(row_labels, row_data)):
    for col in range(T_SHOW):
        ax = axes[row, col]
        ax.imshow(data[col])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(label, color='white', fontsize=10, labelpad=6)
        if row == 0:
            ax.set_title(f't={indices[col]}', color='white', fontsize=8)

plt.suptitle('LeWorldModelRec — reconstructions & prédictions', color='white', fontsize=12)
plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_rec_visuals.png', dpi=120, bbox_inches='tight', facecolor='#111')
plt.show()

In [ ]:
# ── 5b. Post-training — Sauvegarde JSON + Download ────────────────────────────
import json
from datetime import datetime
from pathlib import Path

# Arrêt du monitor GPU
_stop_monitor.set()
_total_time_s = time.time() - _train_start_wall
avg_power_W   = float(np.mean(_power_readings)) if _power_readings else 0.0
energy_Wh     = avg_power_W * _total_time_s / 3600
peak_mem_MB   = torch.cuda.max_memory_allocated() / 1e6
n_params      = sum(p.numel() for p in model.parameters())
n_grad_steps  = len(history['train']) * len(train_loader)
gpu_name      = torch.cuda.get_device_name(0)
gpu_mem_total = torch.cuda.get_device_properties(0).total_memory / 1e6

final_n      = min(5, len(history['val']))
val_tail_std = round(float(np.std(history['val'][-final_n:])), 6) if final_n > 1 else 0.0
tr_tail_std  = round(float(np.std(history['train'][-final_n:])), 6) if final_n > 1 else 0.0

stats = {
    'model':     'rec',
    'timestamp': datetime.now().isoformat(),
    'gpu': {
        'name':         gpu_name,
        'total_mem_MB': round(gpu_mem_total, 0),
    },
    'dataset': {
        'dir':       DATASET_DIR,
        'n_traj':    N_TRAJ,
        'n_frames':  N_FRAMES,
        'img_size':  IMG_SIZE,
        'seq_len':   SEQ_LEN,
        'rollout_k': ROLLOUT_K,
    },
    'hyperparams': {
        'embed_dim':       EMBED_DIM,
        'hidden_dim':      HIDDEN_DIM,
        'lam':             LAM,
        'rec_coef':        REC_COEF,
        'pred_coef':       PRED_COEF,
        'freq_coef':       FREQ_COEF,
        'n_proj':          N_PROJ,
        'batch_size':      BATCH_SIZE,
        'epochs':          EPOCHS,
        'lr_init':         LR,
        'lr_final':        round(history['lr'][-1], 8) if history['lr'] else LR,
        'weight_decay':    WEIGHT_DECAY,
    },
    'time': {
        'total_s':         round(_total_time_s, 1),
        'total_min':       round(_total_time_s / 60, 2),
        'per_epoch_s':     time_per_epoch,
        'avg_per_epoch_s': round(float(np.mean(time_per_epoch)), 2),
        'min_epoch_s':     round(float(np.min(time_per_epoch)), 2),
        'max_epoch_s':     round(float(np.max(time_per_epoch)), 2),
        'gradient_steps':  n_grad_steps,
        'steps_per_sec':   round(n_grad_steps / _total_time_s, 2),
    },
    'memory': {
        'peak_gpu_MB':       round(peak_mem_MB, 1),
        'model_params_M':    round(n_params / 1e6, 3),
        'trainable_params':  n_params,
        'per_epoch_peak_MB': memory_per_epoch_MB,
        'max_per_epoch_MB':  max(memory_per_epoch_MB),
    },
    'energy': {
        'avg_power_W':      round(avg_power_W, 2),
        'total_Wh':         round(energy_Wh, 4),
        'total_J':          round(avg_power_W * _total_time_s, 1),
        'n_power_samples':  len(_power_readings),
        'power_readings_W': [round(p, 1) for p in _power_readings],
    },
    'convergence': {
        'best_val_loss':    round(best_val, 6),
        'best_epoch':       best_epoch,
        'epochs_trained':   len(history['train']),
        'final_train_loss': round(history['train'][-1], 6),
        'final_val_loss':   round(history['val'][-1], 6),
        'overfit_gap':      round(history['val'][-1] - history['train'][-1], 6),
        'val_tail_std':     val_tail_std,
        'train_tail_std':   tr_tail_std,
        'train_loss':    [round(x, 6) for x in history['train']],
        'rec_loss':      [round(x, 6) for x in history['rec']],
        'pred_loss':     [round(x, 6) for x in history['pred']],
        'freq_loss':     [round(x, 6) for x in history['freq']],
        'sigreg':        [round(x, 6) for x in history['sigreg']],
        'val_loss':      [round(x, 6) for x in history['val']],
        'val_rec_loss':  [round(x, 6) for x in history['val_rec']],
        'val_pred_loss': [round(x, 6) for x in history['val_pred']],
        'val_freq_loss': [round(x, 6) for x in history['val_freq']],
        'val_sigreg':    [round(x, 6) for x in history['val_sigreg']],
        'lr_schedule':   [round(x, 8) for x in history['lr']],
    },
}

out_path = f'{VIS_DIR}/training_stats_rec.json'
Path(VIS_DIR).mkdir(exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(stats, f, indent=2)

print(f"GPU      : {gpu_name}")
print(f"Temps    : {_total_time_s/60:.1f} min  |  {n_grad_steps:,} gradient steps")
print(f"Énergie  : {energy_Wh:.3f} Wh  ({avg_power_W*_total_time_s:.0f} J)")
print(f"Mémoire  : {peak_mem_MB:.0f} MB pic  |  {n_params/1e6:.3f} M params")
print(f"Best val : {best_val:.4f}  @ epoch {best_epoch}/{EPOCHS}")
print(f"Overfit  : gap={stats['convergence']['overfit_gap']:+.4f}  val_tail_std={val_tail_std:.4f}")
print(f"JSON  →  {out_path}")

from google.colab import files
files.download(f'{CKPT_DIR}/lewm_rec_best.pt')
files.download(out_path)
print('Done — vous pouvez fermer la session Colab.')

## 6. Visualisations (optionnel)

Les cellules suivantes génèrent les courbes de loss et les reconstructions visuelles.  
Elles ne sont **pas nécessaires** pour récupérer le checkpoint — le téléchargement a déjà eu lieu ci-dessus.